# SECTION 2: CNN-LSTM MODEL TRAINING

In [ ]:

from copy import deepcopy
import os
import pandas as pd
import json

best_val_acc = 0.0
best_model_state = None
best_optimizer_name = ""
results_list = []


In [ ]:

# ================= Final Save Step =================
# Save the best CNN-LSTM model
best_model_path = f"best_cnn_lstm_model_{best_optimizer_name.lower()}.pth"
torch.save(best_model_state, best_model_path)
print(f"✅ Best CNN-LSTM model saved as {best_model_path} with val acc = {best_val_acc:.4f}")

# Save all results to CSV
df_all_results = pd.DataFrame(results_list)
df_all_results.to_csv("cnn_lstm_training_results.csv", index=False)
print("📁 All training results saved to cnn_lstm_training_results.csv")

# Save all results to JSON
with open("cnn_lstm_training_results.json", "w") as json_file:
    json.dump(results_list, json_file, indent=4)
print("📁 All training results saved to cnn_lstm_training_results.json")


In [ ]:
def run_cnn_lstm_attention(optimizer_name="Adam", lr=0.001, epochs=60, patience=10, batch_size=32,
                           split_ratio=(0.8, 0.1, 0.1), model_save_path="best_model.keras"):
    X_train, X_val, X_test, y_train, y_val, y_test = split_data(split_ratio=split_ratio)

    model = build_cnn_lstm_attention(X_train.shape[1:], num_classes=5)

    if optimizer_name == "Adam":
        optimizer = Adam(learning_rate=lr)
    elif optimizer_name == "SGD":
        optimizer = SGD(learning_rate=lr)
    elif optimizer_name == "RMSprop":
        optimizer = RMSprop(learning_rate=lr)
    elif optimizer_name == "Adamax":
        optimizer = Adamax(learning_rate=lr)
    else:
        raise ValueError(f"Unsupported optimizer: {optimizer_name}")

    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

    checkpoint = ModelCheckpoint(model_save_path, monitor='val_accuracy', save_best_only=True, verbose=1)
    early_stop = EarlyStopping(monitor='val_accuracy', patience=patience, restore_best_weights=True)

    history = model.fit(X_train, y_train, validation_data=(X_val, y_val),
                        epochs=epochs, batch_size=batch_size,
                        callbacks=[early_stop, checkpoint], verbose=1)

    best_model = tf.keras.models.load_model(model_save_path, custom_objects={"AttentionLayer": AttentionLayer})

    train_loss, train_acc = best_model.evaluate(X_train, y_train, verbose=0)
    val_loss, val_acc = best_model.evaluate(X_val, y_val, verbose=0)
    test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)

    print(f"\n🧪 Final Training Accuracy: {train_acc * 100:.2f}%")
    print(f"🧪 Final Validation Accuracy: {val_acc * 100:.2f}%")
    print(f"✅ Final Test Accuracy: {test_acc * 100:.2f}%")

    y_pred = best_model.predict(X_test).argmax(axis=1)
    y_true = y_test.argmax(axis=1)

    print("\n📊 Classification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names).plot(cmap='Blues')
    plt.title("Confusion Matrix")
    plt.grid(False)
    plt.show()

    plt.plot(history.history['accuracy'], label='Train Accuracy')
    plt.plot(history.history['val_accuracy'], label='Val Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.title('Training vs Validation Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.title('Training vs Validation Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    return train_acc, val_acc, test_acc

In [ ]:
# Run experiment with unique model filename
exp_id = 7  # Change this for each experiment
model_filename = f"best_model_exp{exp_id}.keras"

train_acc, val_acc, test_acc = run_cnn_lstm_attention(
    optimizer_name="Adam",
    lr=0.001,
    epochs=60,
    patience=10,
    batch_size=32,
    model_save_path=model_filename
)